<a href="https://colab.research.google.com/github/ingridsfrosa/ingridsfrosa.github.io/blob/main/trabalho_bigdata_ingrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comandos para realização do trabalho da matéria de Big Data com uso da biblioteca PySpark.

Este notebook foi projetado para guiar os alunos na realização das práticas de Big Data utilizando PySpark. Certifique-se de seguir cada etapa cuidadosamente para garantir a correta execução das atividades.

Seu trabalho começará na célula 5. Execute as 4 primeiras células para iniciar a atividade.

## <font color=red>Observação importante:</font>

<font color=yellow>Trabalho realizado com uso da biblioteca pandas não será aceito!</font>

## Upload do arquivo `imdb-reviews-pt-br.csv` para dentro do Google Colab

Aqui, você fará o download do dataset necessário para as atividades. Certifique-se de que o arquivo foi descompactado corretamente antes de prosseguir.

In [27]:
!wget https://raw.githubusercontent.com/N-CPUninter/Big_Data/main/data/imdb-reviews-pt-br.zip -O imdb-reviews-pt-br.zip
!unzip imdb-reviews-pt-br.zip
!rm imdb-reviews-pt-br.zip

--2026-04-29 02:14:53--  https://raw.githubusercontent.com/N-CPUninter/Big_Data/main/data/imdb-reviews-pt-br.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 49549692 (47M) [application/zip]
Saving to: ‘imdb-reviews-pt-br.zip’

imdb-reviews-pt-br. 100%[===================>]  47.25M   261MB/s    in 0.2s    

2026-04-29 02:14:55 (261 MB/s) - ‘imdb-reviews-pt-br.zip’ saved [49549692/49549692]

Archive:  imdb-reviews-pt-br.zip
replace imdb-reviews-pt-br.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: imdb-reviews-pt-br.csv  


## Instalação manual das dependências para uso do pyspark no Google Colab

Esta etapa garante que todas as bibliotecas necessárias para o PySpark sejam instaladas no Google Colab.

In [28]:
!apt-get install -y openjdk-17-jdk-headless
!pip install pyspark

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk-headless is already the newest version (17.0.18+8-1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


## Importar, instanciar e criar a SparkSession

A SparkSession é o ponto de entrada para usar o PySpark. Certifique-se de configurar corretamente o nome do aplicativo e o master.

In [29]:
import setuptools
import os
from pyspark.sql import SparkSession

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["JAVA_HOME"], "bin")


appName = "PySpark Trabalho de Big Data"
master = "local"

spark = SparkSession.builder.appName(appName).master(master).getOrCreate()

## Criar spark dataframe do CSV utilizando o método read.csv do spark

Não altere este código e use o dataframe imdb_df criado aqui em todo o seu trabalho. A criação de um dataframe diferente deste poderá causar erros na coluna sentiment e isso refletirá em erros de resposta das questões.

In [30]:
imdb_df = spark.read.csv('imdb-reviews-pt-br.csv',
                         header=True,
                         quote="\"",
                         escape="\"",
                         encoding="UTF-8")

# Questão 1

Nesta questão, você irá calcular a soma dos IDs para entradas onde o sentimento ('sentiment') é 'neg'.

### Objetivo:
- Usar a coluna 'sentiment' como chave e somar os valores da coluna 'id'.

## Criar funções de MAP:
- Criar função para mapear o "sentiment" como chave e o "id" como valor do tipo inteiro

A função map irá transformar cada linha do dataframe em uma **tupla** (chave-valor), onde:
- Chave: coluna 'sentiment'
- Valor: coluna 'id' convertida para inteiro.

In [31]:
# Função Map para transformar cada linha do dataset em um par (Chave, Valor)
def map1(x):
  # Retornamos o sentiment (índice 3) como CHAVE e o id (índice 0) convertido
  # para INT como VALOR
  return (x[3], int(x[0]))

## Cria funções de REDUCE:

- Criar função de reduce para somar os IDs por "sentiment".

A função reduce irá somar os valores dos IDs agrupados por chave ('sentiment').

In [32]:
# Função Reduce para consolidar os dados
def reduceByKey1(x,y):
  # Soma os IDs acumulados (x) com o novo ID da mesma chave (y)
  return (x + y)

## Aplicação do map/reduce e visualização do resultado

Aqui, você aplicará as funções de map e reduce ao dataframe Spark para calcular os resultados. Não se esqueça de usar o método `.collect()` para visualizar os resultados.

In [33]:
# Linha de código para aplicar o map/reduce no dataframe spark
resultado = imdb_df.rdd.map(map1).reduceByKey(reduceByKey1).collect()
# Exibição do resultado da lista de tuplas
print(resultado)

# A variável abaixo pega apenas a primeira tupla da lista (índice 0)
# que corresponde aos dados classificados como 'neg'
resposta_1 = resultado[0]

# Função para mostrar o ID e responder a questão 1.
def mostrar_resposta():
  identificacao = "Ingrid Rosa - RU 4265421"
  print(f"Resposta Questão 1: {resposta_1}")
  print(identificacao)
# Chamada da função para gerar a saída final
mostrar_resposta()

[('neg', 459568555), ('pos', 763600041)]
Resposta Questão 1: ('neg', 459568555)
Ingrid Rosa - RU 4265421


# Questão 2:

Nesta questão, você irá calcular a diferença no número total de palavras entre textos negativos em português e inglês.

### Objetivo:
- Contar as palavras em cada idioma (colunas 'text_pt' e 'text_en') para entradas onde o sentimento ('sentiment') é 'neg'.
- Subtrair o total de palavras em inglês do total em português.

## Criar funções de MAP:
- Criar função para mapear o "sentiment" como chave de uma tupla principal e como valor uma outra tupla com a soma das palavras de cada idioma como valor.

A função map irá transformar cada linha do dataframe em uma tupla (chave-valor), onde:
- Chave: coluna 'sentiment'
- Valor: Nova tupla com:
  - Elemento 0: soma das palavras da coluna 'text_en'
  - Elemento 1: soma das palavras da coluna 'text_pt'

OU
- Chave: coluna 'sentiment'
- Valor: (soma das palavras da coluna 'text_pt') - (soma das palavras da coluna 'text_en')
  

Para contar as palavras deve-se primeiro separar os textos em uma lista de palavras para então descobrir o tamanho desta lista.
Dicas:

1. Use o método .split() e não .split(" ") de string para separar as palavras em uma lista ou use a função split(coluna de texto, regex) do pyspark com o regex igual à "[ ]+" ou "\s+"
2. Use len() para descobrir o tamanho da lista de palavras.

In [34]:
# Função Map para transformar cada linha do dataset em um par (Chave, Valor)
def map2(x):
  # Método .split() quebra o texto em palavras e len() conta a quantidade
  contagem_en = len(x[1].split())
  contagem_pt = len(x[2].split())
  # Retorna a tupla separada por sentiment
  return (x[3], (contagem_en, contagem_pt))

## Cria funções de REDUCE:

- Criar função de reduce para somar o numero de palavras de cada texto português e inglês por "sentiment" (dependerá de como você optou por fazer sua função map2).

A função reduce irá somar os valores das quantidades de palavras agrupados por chave ('sentiment').

In [35]:
# Função Reduce para acumulação dos totais por idioma
def reduceByKey2(x,y):
  # Soma inglês com inglês (índice 0) e português com português (índice 1)
  soma_en = x[0] + y[0]
  soma_pt = x[1] + y[1]
  # Retorna a tupla com o total acumulado de cada idioma
  return (soma_en, soma_pt)

## Aplicação do map/reduce e visualização do resultado

1. Aplicar o map/reduce no seu dataframe spark e realizar o collect() ao final
2. Selecionar os dados referentes aos textos negativos para realizar a subtração.
3. Realizar a subtração das contagens de palavras dos textos negativos para obter o resultado final

In [36]:
# Linha de código para aplicar o map/reduce no seu dataframe spark
resultado_2 = imdb_df.rdd.map(map2).reduceByKey(reduceByKey2).collect()
# Exibição do resultado da lista de tuplas
print(resultado_2)
# Extração apenas da tupla de sentiment negativos
resposta_2 = resultado_2[0]
# Função para mostrar o ID e responder a questão 2.
def mostrar_resposta2():
  identificacao2 = "Ingrid Rosa - RU 4265421"
  # Permite acessar os números dentro da tupla para a subtração
  total_en = resposta_2[1][0]
  total_pt = resposta_2[1][1]
  # Cálculo da diferença final entre idiomas
  diferenca = total_pt - total_en

  print(f"Resposta Questão 2: {diferenca}")
  print(identificacao2)
# Chamada da função para gerar a saída final
mostrar_resposta2()

[('neg', (5400324, 5455273)), ('pos', (5414747, 5462204))]
Resposta Questão 2: 54949
Ingrid Rosa - RU 4265421
